<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=352012350" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=351129098" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=350095850" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PPE YOLOv8 training (Colab / Kaggle)

Use **this notebook** for the E0–E4 grid and for the **vest specialist** (`EXP = "vest_specialist"`, the fix for the weak `no_vest` class). Local 8GB GPUs are only for baseline val, export, and `--batch 8` smokes.

Product bars: **vest / no_vest 95%+**, helmets next, goggles ~70% OK. **Boots are not in this cycle.**

1. Runtime → GPU (Colab) or GPU accelerator (Kaggle), with Internet on.
2. Add secret `ROBOFLOW_API_KEY` (Colab userdata / Kaggle Add-ons → Secrets).
3. Set `EXP` in the config cell (default `vest_specialist`), then Run all cells.
4. The **smoke test** cell runs first (~3-5 min). If it raises, stop and fix — the long run would have failed the same way.
5. The **sanity check** cell after training compares independent test mAP50 to training-time val mAP50 and aborts on a mismatch.
6. Cell 2 prints the checked-out commit — confirm it is the one you pushed before spending any GPU time.

Prefer **one** of Colab or Kaggle, not both.

In [1]:
EXP = "vest_specialist"

In [2]:
import os
from pathlib import Path

# Colab secret, then Kaggle, then env (local fallback).
try:
    from google.colab import userdata
    os.environ.setdefault("ROBOFLOW_API_KEY", userdata.get("ROBOFLOW_API_KEY"))
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("ROBOFLOW_API_KEY", UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    except Exception:
        pass

assert os.environ.get("ROBOFLOW_API_KEY"), "Set ROBOFLOW_API_KEY as a Colab/Kaggle secret"
print("key_set", "colab" if IN_COLAB else "kaggle_or_local")

REPO = Path("/content/ppe") if IN_COLAB else Path("/kaggle/working/ppe")
if (REPO / "scripts" / "train.py").exists():
    # Repo already checked out from a previous run in this session (Kaggle/Colab
    # keep /kaggle/working and /content between cell reruns) — pull latest instead
    # of silently training against a stale, possibly-already-fixed-upstream copy.
    print(f"{REPO} already exists — pulling latest instead of re-cloning")
    !git -C {REPO} fetch --depth 1 origin main
    !git -C {REPO} reset --hard origin/main
else:
    REPO.mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git {REPO}
os.chdir(REPO)
print("cwd", Path.cwd())
!git -C {REPO} log -1 --oneline

key_set kaggle_or_local
Cloning into '/kaggle/working/ppe'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 169 (delta 5), reused 124 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 49.78 MiB | 37.02 MiB/s, done.
Resolving deltas: 100% (5/5), done.
cwd /kaggle/working/ppe
69167fe (grafted, HEAD -> main, origin/main, origin/HEAD) Kaggle Notebook | Fixe output write script; Vest specialize | Version 6


In [3]:
# ultralytics is pinned to the version the configs/scripts were validated against
# (get_cfg rejects unknown kwargs, so version drift can break a run mid-setup).
%pip install -q "ultralytics==8.4.37" roboflow pyyaml opencv-python-headless
%pip install -q -e .
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "No GPU attached - enable the GPU accelerator before spending session time."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 75.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ppe (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
cuda True Tesla T4


In [4]:
# >>> The ONLY place to choose the experiment <<<
#   vest_specialist : 2-class (vest, no_vest) model that lifts no_vest (~40 min total, ~20 min GPU)
#   e0_n | e1_s | e2_focal | e3_augs : 14-class runs on the 12k subset (hours)
#   e4_full44k      : 14-class on the full 44k (5-6 h)
# The data prep, smoke test, long run and sanity check below all read these variables, so they
# cannot disagree about what is being trained.
EXP = "vest_specialist"

import sys
sys.path.insert(0, "scripts")
LOG_DIR = "/content" if IN_COLAB else "/kaggle/working"
SPECIALIST = EXP == "vest_specialist"
# The specialist has its own 2-class dataset, so it is scored on that dataset's clean (deduplicated)
# test split; the 14-class runs are scored on data/processed/combined (sanity_check's default).
SANITY_DATA = "data/processed/vest_specialist/data.yaml" if SPECIALIST else None
print(f"EXP={EXP}  specialist={SPECIALIST}  sanity_data={SANITY_DATA}")

EXP=vest_specialist  specialist=True  sanity_data=data/processed/vest_specialist/data.yaml


In [5]:
if SPECIALIST:
    # Vest specialist: only what it needs (~10 min less quota than the 14-class prep). Every step
    # reads the REMAPPED data (data/processed/...), never data/raw labels.
    !python scripts/download_datasets.py --execute --only combined gap_vest
    !python scripts/remap_labels.py --source data/raw/combined --out data/processed/combined --mapping combined
    !python scripts/remap_labels.py --source data/raw/gap_vest --out data/processed/gap_vest --mapping gap_vest
    # Images where vest status was actually annotated (84% of worker images have none).
    !python scripts/make_class_subset.py --classes vest no_vest --out data/raw/vest_annotated
    # Drop test images that near-duplicate a train/valid image, so the test score can't be memorization.
    !python scripts/dedupe_test.py --subset data/raw/vest_annotated/test.txt --out-list data/raw/vest_annotated/test_clean.txt --out-yaml data/raw/vest_annotated/data_clean.yaml
    !python scripts/make_vest_specialist_data.py
    # Early gate (local reference: train 3253 / valid 834 / test 145): fail here, not after training.
    counts = {s: len(list(Path(f"data/processed/vest_specialist/{s}/images").glob("*"))) for s in ("train", "valid", "test")}
    print("specialist images per split:", counts)
    assert counts["train"] > 3000 and counts["valid"] > 700 and counts["test"] > 100, f"specialist dataset looks wrong: {counts}"
else:
    # Combined (~2.4GB zip) then Hard Hat Universe. Construction is optional for mapped eval.
    !python scripts/download_datasets.py --execute --only combined hardhat
    !python scripts/remap_labels.py --source data/raw/combined --out data/processed/combined --mapping combined
    !python scripts/remap_labels.py --source data/raw/hardhat --out data/processed/hardhat --mapping hhu
    !python scripts/make_subset.py --source data/processed/combined --out data/raw/combined_12k --n 12000 --seed 42
    !python scripts/analyze_distribution.py

Requesting export roboflow-universe-projects/personal-protective-equipment-combined-model/4/yolov8 -> /kaggle/working/ppe/data/raw/combined
  zip 100.0% [2542363062/2542363062 bytes]
Extracting combined.zip -> /kaggle/working/ppe/data/raw/combined
Downloaded combined to /kaggle/working/ppe/data/raw/combined
Requesting export novest/no-vest-detect/1/yolov8 -> /kaggle/working/ppe/data/raw/gap_vest
  zip 100.0% [27051865/27051865 bytes]
Extracting gap_vest.zip -> /kaggle/working/ppe/data/raw/gap_vest
Downloaded gap_vest to /kaggle/working/ppe/data/raw/gap_vest
train: 30765 images, dropped 0 unmapped boxes
valid: 8814 images, dropped 0 unmapped boxes
test: 4423 images, dropped 0 unmapped boxes
Wrote remapped dataset to /kaggle/working/ppe/data/processed/combined
train: 873 images, dropped 0 unmapped boxes
valid: 100 images, dropped 0 unmapped boxes
test: 25 images, dropped 0 unmapped boxes
Wrote remapped dataset to /kaggle/working/ppe/data/processed/gap_vest
train: 2728 / 30765 images  box

In [6]:
# SMOKE TEST (~3-5 min on a T4): the same pipeline the long run uses - 1 epoch on 2% of the
# train set, then the same sanity check. It exists to catch plumbing failures (paths, kwargs,
# missing checkpoint, eval/parse errors) BEFORE the long run, and it exercises EXP's own data
# path (the specialist's 2-class dataset, the 12k subset list for E0-E3, the full directory yaml
# for E4). Any exception here aborts "Run All" - fix it before spending credits. A
# "WARNING: ... under-trained" line is expected and fine; a failure is not.
print(f"Experiment: {EXP}. Resolved data/epochs (eyeball that data is what you expect):")
!python scripts/train.py --exp {EXP} --dry-run | grep -E "data:|epochs:|batch:"
!python scripts/train.py --exp {EXP} --name smoke --device 0 --batch 16 --epochs 1 --fraction 0.02 > {LOG_DIR}/train_smoke.log 2>&1
!tail -n 15 {LOG_DIR}/train_smoke.log
import sanity_check
sanity_check.run("smoke", data=SANITY_DATA)

Experiment: vest_specialist. Resolved data/epochs (eyeball that data is what you expect):
  batch: 16
  data: /kaggle/working/ppe/data/processed/vest_specialist/data.yaml
  epochs: 30
                   all        834       1703    0.00402      0.573     0.0236    0.00581

1 epochs completed in 0.004 hours.
Optimizer stripped from /kaggle/working/ppe/runs/train/smoke/weights/last.pt, 6.2MB
Optimizer stripped from /kaggle/working/ppe/runs/train/smoke/weights/best.pt, 6.2MB

Validating /kaggle/working/ppe/runs/train/smoke/weights/best.pt...
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 4.4it/s 6.2s
                   all        834       1703    0.00403      0.576     0.0237     0.0058
                  vest        614       1292    0.00626      0.567     0.038

'warn'

In [7]:
# Long run for EXP (set in the config cell above).
# P100/T4: batch 16. If OOM, add --batch 8. Kaggle GPU sessions cap at ~9h.
# Measured on a T4: E0 on the 12k subset = 94 epochs in 2.97h (~112 s/epoch, early-stopped).
# vest_specialist: ~3.2k images, nano, 30 epochs (patience 10) -> roughly 15-20 min on a T4
# (its CPU run reached val mAP50 0.88 by epoch 14 of 15, still rising).
# The full 44k set is ~3.6x the 12k images, so e4_full44k (50 epochs, patience 15) is roughly
# 5-6h plus ~20 min of download/remap - inside the cap, but watch the clock.
#
# IMPORTANT: redirect to a log file instead of letting output stream into this
# cell. A 100-epoch run's carriage-return-updating progress bars, captured
# verbatim into the notebook's cell-output JSON, can bloat that JSON to the
# point where Kaggle's post-run nbconvert step (which always runs, converting
# the executed notebook to .ipynb/.html for the Output tab) takes HOURS to
# process it - the kernel looks "still running" and keeps billing the whole
# time, even though training itself finished long before. Confirmed exactly
# this happened on 2026-09-15/16: nbconvert's regex-based cell-output
# processing (mistune.py / filter_links.py) took ~2.8 hours on one bloated
# cell alone. Redirecting keeps this cell's own output tiny (just the tail)
# while the full log still lands on disk.
LOG = f"{LOG_DIR}/train_{EXP}.log"
!python scripts/train.py --exp {EXP} --device 0 --batch 16 > {LOG} 2>&1
print(f"Full log: {LOG}")
!tail -n 60 {LOG}

Full log: /kaggle/working/train_vest_specialist.log
      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      22/30      2.55G     0.9431     0.7598      1.207          7        640: 100% ━━━━━━━━━━━━ 204/204 6.0it/s 34.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 5.7it/s 4.7s
                   all        834       1703      0.833       0.81      0.873      0.562

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      23/30      2.55G     0.9319     0.7476      1.191         20        640: 100% ━━━━━━━━━━━━ 204/204 5.9it/s 34.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 5.8it/s 4.7s
                   all        834       1703      0.854      0.826      0.881       0.57

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      24/30      2.55G     0.9015     0.70

In [8]:
# Sanity-check the just-trained checkpoint on the held-out test split RIGHT NOW, while
# GPU/credits are still available. Logic lives in scripts/sanity_check.py (unit-tested):
# it compares the independent test mAP50 against the run's own training-time val mAP50.
# A large gap means training and eval used different labels (seen twice: an unremapped
# data path in configs/data/combined.yaml, then make_subset.py resolving symlinks back to
# raw/). It raises AssertionError on a mismatch - do NOT copy the checkpoint off the VM then.
# For the specialist this scores its 2-class clean test split (SANITY_DATA), where the local CPU
# run reference is vest ~0.86 / no_vest ~0.74 mAP50 (E0: 0.73 / 0.19, Hexmon: 0.78 / 0.41).
import sanity_check
sanity_check.run(EXP, data=SANITY_DATA)

Evaluating /kaggle/working/ppe/runs/train/vest_specialist/weights/best.pt on /kaggle/working/ppe/data/processed/vest_specialist/data.yaml split=test
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1108.5±616.8 MB/s, size: 48.5 KB)

val: Scanning /kaggle/working/ppe/data/processed/vest_specialist/test/labels.cache... 145 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 145/145 22.5Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 10% ━─────────── 1/10 1.4it/s 0.2s<6.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 20% ━━────────── 2/10 3.4it/s 0.3s<2.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 30% ━━━╸──────── 3/10 4.8it/s 0.5s<1.5s
                 Class     Images  Instances   

'pass'

The sanity-check cell above already confirms the scores look real before you leave the VM. If it passed, copy these off the VM (Kaggle Output panel; they are small):
`runs/train/{EXP}/weights/best.pt`, `runs/train/{EXP}/results.csv`, `runs/train/{EXP}/args.yaml`, `results/analysis/eval_{EXP}.json` (with `EXP` as set above).
For `vest_specialist`, score it locally on the clean test, pick `PPE_CLASS_CONF` with `scripts/diagnose_class.py`, and deploy it via `PPE_SPECIALIST` (see `docs/edge_npu.md`).